# 01 · Exploración de fuentes — Tikal y Flores, Petén
**Observatorio de Inteligencia Turística · Grupo 2 · Osman de León**

Objetivo: reconocer la estructura de las fuentes oficiales entregadas, comprobar calidad básica y revisar los resultados del primer ETL. Este notebook **no entrena modelos** y no publica microdatos. Ejecutar desde el repositorio con los archivos en `data/raw/`.

> **Criterios metodológicos:** la movilidad diaria representa presencia, no visitantes únicos mensuales; los subdestinos se pueden solapar y no se suman para estimar el total de Petén; los pasajeros del AIMM no equivalen a turistas; los datos faltantes no son ceros.

## 1. Preparación y rutas

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

# Funciona tanto al abrir el notebook desde notebooks/ como desde la raíz.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir() and (ROOT.parent / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('Abre Jupyter desde la raíz del repositorio o desde notebooks/.')
sys.path.insert(0, str(ROOT))
from src import etl
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
print('Repositorio:', ROOT)
print('Archivos originales:', len(list(RAW.glob('*'))))
print('Archivos procesados:', len(list(PROCESSED.glob('*.csv'))))

Repositorio: C:\Users\Emadlg\Desktop\UVG\proyecto3_Data
Archivos originales: 10
Archivos procesados: 5


## 2. Inventario de fuentes originales

In [2]:
inventario = pd.DataFrame([
    {'archivo': p.name, 'tipo': p.suffix.lower(), 'tamano_mb': round(p.stat().st_size/1024**2, 2)}
    for p in sorted(RAW.iterdir()) if p.is_file()
])
display(inventario)
requeridos = [
    ('BD_moviidad_mensual.xlsx', 'BD_movilidad_mensual.xlsx'),
    ('BD_moviidad_diario.xlsx', 'BD_movilidad_diario.xlsx'),
    ('Visitantes_Tikal_2017-2026.xlsx',),
    ('AIMM_Consolidado_2022-2025.xlsx',),
    ('AIMM_Aerolineas_2023-2026.xlsx',),
    ('AIMM_Operaciones_Aeronauticas_2022-2026.xlsx',),
    ('indice_ocupacion_mensual_2023_2025_1.csv',),
    ('oferta_hotelera_2022_2026.csv',),
    ('Base_Migracion_2009-2026jun.xlsx',),
]
faltantes = [opciones for opciones in requeridos if not any((RAW/n).exists() for n in opciones)]
print('Fuentes requeridas faltantes:', faltantes if faltantes else 'Ninguna')

,archivo,tipo,tamano_mb
0,AIMM_Aerolineas_2023-2026.xlsx,.xlsx,0.03
1,AIMM_Consolidado_2022-2025.xlsx,.xlsx,0.02
2,AIMM_Operaciones_Aeronauticas_2022-2026.xlsx,.xlsx,0.04
3,Base_Migracion_2009-2026jun.xlsx,.xlsx,9.29
4,BD_moviidad_diario.xlsx,.xlsx,8.23
5,BD_moviidad_mensual.xlsx,.xlsx,1.06
6,Encuesta_visitante_Peten_2025.xlsx,.xlsx,0.42
7,indice_ocupacion_mensual_2023_2025_1.csv,.csv,0.02
8,oferta_hotelera_2022_2026.csv,.csv,0.01
9,Visitantes_Tikal_2017-2026.xlsx,.xlsx,0.02


Fuentes requeridas faltantes: Ninguna


## 3. Inspección de la movilidad de Petén

Los libros de movilidad contienen una anomalía de metadatos (`dimension=A1`). La función `leer_hoja_movilidad` usa `openpyxl.reset_dimensions()` para recuperar todas las filas sin modificar el archivo original.

In [3]:
mensual = etl.leer_hoja_movilidad(etl.encontrar(RAW, 'BD_moviidad_mensual.xlsx','BD_movilidad_mensual.xlsx'))
diario = etl.leer_hoja_movilidad(etl.encontrar(RAW, 'BD_moviidad_diario.xlsx','BD_movilidad_diario.xlsx'))
for nombre, df in [('Mensual',mensual), ('Diaria',diario)]:
    print(f'\n{nombre}: {df.shape[0]:,} filas, {df.shape[1]} columnas')
    print('Columnas:', list(df.columns))
    print('Duplicados exactos:', int(df.duplicated().sum()))
    display(df.isna().sum().sort_values(ascending=False).head(8).to_frame('nulos'))
    display(df.head(3))

c:\Users\Emadlg\Desktop\UVG\proyecto3_Data\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



Mensual: 3,473 filas, 10 columnas
Columnas: ['destino', 'sub_grupo', 'anio', 'mes', 'pais', 'es_consecutivo', 'fecha', 'visitantes', 'dias_persona', 'permanencia_prom']
Duplicados exactos: 0


,nulos
permanencia_prom,8
pais,8
sub_grupo,0
destino,0
mes,0
anio,0
es_consecutivo,0
fecha,0


,destino,sub_grupo,anio,mes,pais,es_consecutivo,fecha,visitantes,dias_persona,permanencia_prom
0,El Remate Y Alrededores Lago,Peten,2025.0,9.0,Alemania,mayor a 1 dia,2025-09-01,17.0,38.0,2.235294
1,El Remate Y Alrededores Lago,Peten,2025.0,9.0,Alemania,menor a 1 dia,2025-09-01,13.0,13.0,1.000000
2,El Remate Y Alrededores Lago,Peten,2025.0,9.0,Argentina,mayor a 1 dia,2025-09-01,5.0,15.0,3.000000



Diaria: 31,513 filas, 10 columnas
Columnas: ['destino', 'sub_grupo', 'anio', 'mes', 'idfecha', 'dia_semana', 'pais', 'Week', 'evento', 'cantidad_visitantes']
Duplicados exactos: 0


,nulos
pais,10
destino,0
sub_grupo,0
anio,0
idfecha,0
mes,0
dia_semana,0
Week,0


,destino,sub_grupo,anio,mes,idfecha,dia_semana,pais,Week,evento,cantidad_visitantes
0,EL REMATE Y ALREDEDORES LAGO,PETEN,2025.0,4.0,2025-04-11,Viernes,ALEMANIA,15.0,Semana Santa,6.0
1,EL REMATE Y ALREDEDORES LAGO,PETEN,2025.0,4.0,2025-04-11,Viernes,ARGENTINA,15.0,Semana Santa,2.0
2,EL REMATE Y ALREDEDORES LAGO,PETEN,2025.0,4.0,2025-04-11,Viernes,AUSTRIA,15.0,Semana Santa,13.0


## 4. Cobertura temporal y subdestinos

In [4]:
for nombre, df, fecha in [('Mensual', mensual, 'fecha'),('Diaria', diario, 'idfecha')]:
    fechas = pd.to_datetime(df[fecha], errors='coerce')
    print(nombre, '| desde:', fechas.min(), '| hasta:', fechas.max(), '| fechas distintas:', fechas.nunique())
    display(df['destino'].value_counts().to_frame('registros'))
print('Segmentos mensuales:')
display(mensual['es_consecutivo'].value_counts().to_frame('registros'))
print('Eventos diarios:')
display(diario['evento'].value_counts(dropna=False).head(12).to_frame('registros'))

Mensual | desde: 2025-09-01 00:00:00 | hasta: 2026-08-01 00:00:00 | fechas distintas: 10


,registros
destino,
Mundo Maya Aeropuerto,1203
El Remate Y Alrededores Lago,1136
Isla De Flores,1134


Diaria | desde: 2025-04-11 00:00:00 | hasta: 2026-09-15 00:00:00 | fechas distintas: 324


,registros
destino,
MUNDO MAYA AEROPUERTO,11237
ISLA DE FLORES,10356
EL REMATE Y ALREDEDORES LAGO,9920


Segmentos mensuales:


,registros
es_consecutivo,
menor a 1 dia,1802
mayor a 1 dia,1671


Eventos diarios:


,registros
evento,
Regular,25067
Semana Santa,2096
Fiestas Agostinas,1022
Festival de las Flores,996
Festival de las Luces,825
Día del Ejército,726
20 de Octubre,325
Año Nuevo,219
15 de Septiembre,139


## 5. Otras fuentes: hojas y encabezados

Se listan las hojas y dimensiones de los Excel. La inspección de datos detallados se limita a las columnas necesarias para el ETL. No mostrar ni exportar identificadores de encuestados.

In [5]:
from openpyxl import load_workbook
nombres = ['Visitantes_Tikal_2017-2026.xlsx','AIMM_Consolidado_2022-2025.xlsx',
           'AIMM_Aerolineas_2023-2026.xlsx','AIMM_Operaciones_Aeronauticas_2022-2026.xlsx',
           'Base_Migracion_2009-2026jun.xlsx']
filas = []
for nombre in nombres:
    ruta = RAW / nombre
    if not ruta.exists():
        continue
    wb = load_workbook(ruta, read_only=True, data_only=True)
    for ws in wb:
        filas.append({'archivo':nombre,'hoja':ws.title,'filas_declaradas':ws.max_row,'columnas_declaradas':ws.max_column})
    wb.close()
display(pd.DataFrame(filas))

,archivo,hoja,filas_declaradas,columnas_declaradas
0,Visitantes_Tikal_2017-2026.xlsx,Datos,117,7
1,AIMM_Consolidado_2022-2025.xlsx,Base Consolidada,219,6
2,AIMM_Consolidado_2022-2025.xlsx,Resumen Anual,31,5
3,AIMM_Aerolineas_2023-2026.xlsx,Base de Datos,505,7
4,AIMM_Aerolineas_2023-2026.xlsx,Resumen Aerolínea,23,6
5,AIMM_Aerolineas_2023-2026.xlsx,Tendencia Mensual,65,6
6,AIMM_Operaciones_Aeronauticas_2022-2026.xlsx,Base de Operaciones,771,6
7,AIMM_Operaciones_Aeronauticas_2022-2026.xlsx,Aux (Cruces y Poptún),79,5
8,AIMM_Operaciones_Aeronauticas_2022-2026.xlsx,Resumen Anual,18,6
9,AIMM_Operaciones_Aeronauticas_2022-2026.xlsx,Operador por Año,14,7


## 6. Ejecutar el ETL reproducible

In [6]:
salidas = etl.ejecutar(RAW, PROCESSED)
resumen = []
for nombre, df in salidas.items():
    col_fecha = 'fecha' if 'fecha' in df.columns else None
    resumen.append({'archivo':nombre,'filas':len(df),'columnas':len(df.columns),
                    'fecha_min':str(df[col_fecha].min())[:10] if col_fecha else 'No aplica',
                    'fecha_max':str(df[col_fecha].max())[:10] if col_fecha else 'No aplica',
                    'duplicados_exactos':int(df.duplicated().sum())})
display(pd.DataFrame(resumen))

c:\Users\Emadlg\Desktop\UVG\proyecto3_Data\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Emadlg\Desktop\UVG\proyecto3_Data\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,archivo,filas,columnas,fecha_min,fecha_max,duplicados_exactos
0,movilidad_mensual_peten.csv,60,6,2025-09-01,2026-08-01,0
1,movilidad_diaria_peten.csv,972,4,2025-04-11,2026-09-15,0
2,tikal_mensual.csv,116,6,2017-01-01,2026-08-01,0
3,aimm_pasajeros_mensual.csv,248,5,2022-01-01,2026-04-01,0
4,aimm_operaciones_mensual.csv,104,3,2022-01-01,2026-04-01,0
5,ocupacion_peten.csv,36,2,2023-01-01,2025-12-01,0
6,oferta_hotelera_peten.csv,15,6,No aplica,No aplica,0
7,migracion_nacional_mensual.csv,210,2,2009-01-01,2026-06-01,0
8,comparacion_fuentes_peten.csv,6,4,2025-09-01,2026-04-01,0


## 7. Verificaciones metodológicas

In [7]:
m = salidas['movilidad_mensual_peten.csv']
t = salidas['tikal_mensual.csv']
a = salidas['aimm_pasajeros_mensual.csv']
o = salidas['ocupacion_peten.csv']
assert (m['visitantes'] >= 0).all()
assert (m['dias_persona'] >= m['visitantes']).all()
assert (t[['nacionales','extranjeros','otros']].sum(axis=1) == t['total']).all()
assert o['indice_ocupacion'].between(0,1).all()
assert not o['fecha'].duplicated().any()
print('Validaciones básicas superadas.')
print('Meses con cero en Tikal (revisar antes de interpretar):')
display(t.loc[t.total.eq(0), ['fecha','total']])
print('Fuentes AIMM usadas por año:')
display(a.assign(anio=pd.to_datetime(a.fecha).dt.year).groupby(['anio','fuente']).size().to_frame('filas'))

Validaciones básicas superadas.
Meses con cero en Tikal (revisar antes de interpretar):


,fecha,total
0,2017-01-01,0
39,2020-04-01,0
40,2020-05-01,0
41,2020-06-01,0
42,2020-07-01,0
43,2020-08-01,0
44,2020-09-01,0


Fuentes AIMM usadas por año:


,,filas
anio,fuente,
2022,consolidado,72
2023,aerolineas,24
2024,consolidado,72
2025,consolidado,72
2026,aerolineas,8


## 8. Visualizaciones exploratorias

Las siguientes gráficas describen fuentes **diferentes**. No sumar pasajeros, boletos y movilidad; tampoco sumar subdestinos de telefonía para obtener visitantes únicos de Petén.

In [8]:
t = t.copy(); t['fecha'] = pd.to_datetime(t.fecha)
fig = px.line(t, x='fecha', y=['nacionales','extranjeros','otros'],
              title='Tikal: boletos registrados por categoría', labels={'value':'Boletos','fecha':'Mes','variable':'Categoría'})
fig.show()

In [9]:
o = o.copy(); o['fecha'] = pd.to_datetime(o.fecha)
fig = px.line(o, x='fecha', y='indice_ocupacion', markers=True,
              title='Petén: índice de ocupación hotelera', labels={'indice_ocupacion':'Ocupación (proporción)','fecha':'Mes'})
fig.show()

In [10]:
m = m.copy(); m['fecha'] = pd.to_datetime(m.fecha)
sub = sorted(m.subdestino.unique())[0]
subdf = m[m.subdestino.eq(sub)].groupby('fecha',as_index=False).visitantes.sum()
fig = px.bar(subdf, x='fecha', y='visitantes', title=f'Telefonía: visitantes mensuales en {sub}',
             labels={'visitantes':'Visitantes del polígono','fecha':'Mes'})
fig.show()
print('Este gráfico es de UN subdestino; no representa el total único de Petén.')

Este gráfico es de UN subdestino; no representa el total único de Petén.


## 9. Hallazgos y próximos pasos

- Los datos originales se conservan intactos y las salidas se generan en `data/processed/`.
- Los registros de telefonía diaria representan **presencia diaria**; no sumarlos para construir el total mensual.
- La estadía media se calcula como **días-persona / visitantes** únicamente para el segmento `mayor a 1 dia`.
- La cobertura de telefonía tiene meses ausentes: mostrar **Sin dato** y no cero.
- Tikal contiene meses con cero que requieren interpretación documental.
- El AIMM tiene fuentes con coberturas distintas; el ETL elige una fuente por mes para evitar duplicaciones.
- El siguiente notebook (`02_limpieza_validacion.ipynb`) profundizará en calidad, cruces y reglas de negocio; después construiremos indicadores y pronósticos.

**Confidencialidad:** mantener el repositorio privado; no subir datos originales ni derivados restringidos. No guardar salidas del notebook con microdatos.